# 09 · How do we plug a vision model into a real language model?

In Notebook 08 the model could only pick from a fixed list of answers. Now we let a real language model *write* the answer. Following the LLaVA paper, [*Visual Instruction Tuning*](https://arxiv.org/abs/2304.08485), we'll plug a frozen CLIP into a small Qwen model with a single linear layer and train it in the paper's two stages.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 4, Section 4.1.4, *Open-Source Visual Assistants: LLaVA*, and Chapter 3, Section 3.2.1, *Prefix and Prompt-Based Conditioning*.

> **Heads up: this one's the big one.** It needs a GPU, downloads about 13.5 GB, wants 35–50 GB of free disk, and Stage 2 takes several hours. Start it, then go do something else for a while.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with. None of them retrain anything; they only reuse the data, the cached features, and the model you already have.

## 1. What exactly are we building?

Here's the question we'll keep coming back to: **where do the image patches enter the language model, and which weights actually learn from the answer?** Every section answers one piece of it.

Here's how the notebook lines up with the paper, so you can read the two side by side:

| Notebook section | What happens | LLaVA paper |
|---|---|---|
| 3. The multimodal interface | frozen CLIP, plus a linear projector into Qwen's embedding space | Section 4.1, Architecture |
| 4. The feature cache | penultimate-layer CLIP patches, computed once per image | Section 4.1 (which grid features to use) |
| 5. Sequences and labels | splice patches into the prompt, put loss only on answer tokens | Section 4.2, Training |
| 6. Stage 1 | train only the projector, on captions | Section 4.2, "Stage 1: Pre-training for Feature Alignment" |
| 7. Stage 2 | train projector and language model, on VQAv2 | Section 4.2, "Stage 2: Fine-tuning End-to-End" |
| 8. Held-out checks | generate answers, swap images | our own sanity checks, not from the paper |

Training on complete splits also brings a practical problem. Re-encoding an image for every single question would waste most of the run. So we'll run frozen CLIP once per photo, save the patches to disk, and look them up by row number.

### 1.1 Pick a GPU (before PyTorch shows up)

Set `GPU_ID` in the next cell to the GPU you want (`nvidia-smi` lists them; leave it at `"0"` if you only have one). Unlike the earlier notebooks, this one really does need a GPU, so CPU-only isn't an option.

CUDA reads these settings when it starts up, so this cell has to run before torch is imported. Whichever GPU you pick shows up inside this process as `cuda:0`. If you restart the kernel, run this cell first again.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   (CPU-only isn't an option here: this notebook needs a GPU)
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

### 1.2 Grab our tools

Four kinds of tools here:

- `datasets` to load the data and store our feature cache in **Arrow**, the on-disk table format `datasets` uses. It can read just the rows you ask for, without pulling the whole file into memory.
- PyTorch for training.
- `transformers` for the pretrained CLIP and Qwen models.
- Matplotlib and `tqdm` for plots and progress bars.

Nothing gets downloaded yet.

In [ ]:
from collections import OrderedDict  # data + Arrow storage, training, pretrained models, plots and progress bars
from pathlib import Path
from textwrap import fill
import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Array2D, Features, Value, load_dataset, load_from_disk
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, CLIPImageProcessor, CLIPModel

### 1.3 Same two stages as the paper, smaller models

One word to be careful with: when this notebook says **full**, it means every row of the training splits we picked. It doesn't mean the original LLaVA data mix, and it definitely doesn't mean its model size. Here's how we compare:

| Decision | Original LLaVA | This notebook |
|---|---|---|
| Vision tower | CLIP ViT-L/14 | CLIP ViT-B/32 |
| Language model | Vicuna 13B; 7B ablation | Qwen2.5 0.5B Instruct |
| Vision-to-language bridge | Linear projection | Linear projection |
| Stage 1 purpose | Align image features with language embeddings | Same purpose |
| Stage 1 data | 595K filtered CC3M pairs | All 5,000 selected COCO captions |
| Stage 1 updates | Connector only | Connector only |
| Stage 2 purpose | Visual instruction tuning | Same optimization role, narrower task variety |
| Stage 2 data | LLaVA-Instruct-158K | All 443,757 VQAv2 turns |
| Stage 2 updates | Connector and language model | Connector and language model |

VQAv2 mostly asks for short factual answers (think "yes", "2", or "red"). That makes it a real, large instruction source, but it's nowhere near the paper's long conversations and reasoning data. Keep that in mind when you judge anything this model says.

### 1.4 Pin the exact model versions

The two pretrained models set the rules everything else has to follow. CLIP decides how many patches we get and how wide they are. Qwen decides the token IDs and how wide its word embeddings are. Pinning exact revisions (specific commits on the Hugging Face Hub) means those rules can't quietly change between runs.

In [ ]:
CLIP_ID = "openai/clip-vit-base-patch32"  # pinned revisions, so neither interface can shift under us
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
LM_ID = "Qwen/Qwen2.5-0.5B-Instruct"
LM_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"

Now the data. Stage 1's alignment data is 5,000 COCO captions written for 1,000 images. Several descriptions of the same photo are great supervision, but there's no reason to run CLIP on that photo five times. They'll all share one cached encoding.

In [ ]:
CAPTION_DATASET_ID = "Erland/coco_captions_small"  # every row of this pinned train split goes into Stage 1
CAPTION_REVISION = "a516b5101cec691eb65f455ffdb054afb2fdd5dd"

For Stage 2 we use VQAv2 from **The Cauldron**, a big collection of vision-language datasets on the Hub. It stores VQAv2 grouped by image: one row holds one photo plus a list of its question–answer turns. (A **turn** here is one user question plus one assistant answer.) That's 82,772 image rows and 443,757 turns. We'll train on every turn but encode each photo only once.

In [ ]:
INSTRUCTION_DATASET_ID = "HuggingFaceM4/the_cauldron"  # the vqav2 config holds the complete train split
INSTRUCTION_CONFIG = "vqav2"
INSTRUCTION_REVISION = "847a98a779b1652d65111daf20c972dfcd333605"

And last, a separate 1,000-row sample of VQAv2 *validation* data. This one never touches an optimizer. We'll use it to compare loss before and after Stage 2, and for the image-swap check at the end.

In [ ]:
VALIDATION_DATASET_ID = "Multimodal-Fatima/VQAv2_sample_validation"  # held out: never enters either optimizer
VALIDATION_REVISION = "99487d2651df3799002b2fb3e455741744514a02"

Now the training knobs. One **epoch** means one pass over every training record. Stage 1 gets bigger batches and a bigger learning rate, because the only thing learning is a brand-new linear layer. Stage 2 gets a much smaller learning rate, because now pretrained Qwen weights change too, and we don't want to wreck what Qwen already knows.

> **Heads up:** Changing any of these means retraining, and for Stage 2 that's hours. If you just want to experiment, the TODOs are much cheaper.

In [ ]:
ALIGNMENT_EPOCHS, INSTRUCTION_EPOCHS = 1, 1  # one pass over every training record
ALIGNMENT_BATCH_SIZE, INSTRUCTION_BATCH_SIZE = 32, 16
CACHE_BATCH_SIZE = 64
ALIGNMENT_LR, INSTRUCTION_LR = 1e-3, 5e-6  # the new layer learns fast, pretrained weights learn gently

The seed fixes the projector's starting weights and the shuffle order. The two smoothing widths only change how the loss curves get *drawn*. Every mini-batch still makes its own optimizer update.

In [ ]:
SEED = 9  # same shuffle order and starting projector weights on every run
ALIGNMENT_SMOOTHING, INSTRUCTION_SMOOTHING = 20, 200
torch.manual_seed(SEED)
if not torch.cuda.is_available():  # this notebook needs a GPU, so fail early with a clear message
    raise RuntimeError("No GPU visible. Check GPU_ID in the first code cell.")
device = torch.device("cuda")
print("training device:", device)

### 1.5 Decide where the big files go

The first run creates `artifacts/mini_llava/features` and `artifacts/mini_llava/checkpoints`. If a finished feature cache is already sitting there, later runs reuse it instead of running CLIP again. That's what makes restarts painless.

In [ ]:
project_root = Path.cwd()  # Jupyter might start at the repo root or inside notebooks/
if project_root.name == "notebooks":
    project_root = project_root.parent
artifact_dir = project_root / "artifacts" / "mini_llava"
feature_dir = artifact_dir / "features"
checkpoint_dir = artifact_dir / "checkpoints"
feature_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

The cache names include short pieces of the dataset and model revisions. It's a tiny bit of bookkeeping, but it means features from one CLIP revision can never get mistaken for another's.

In [ ]:
clip_cache_tag = f"clip_b32_penultimate_{CLIP_REVISION[:8]}"  # the name records the encoder, its revision, and the layer
caption_cache_path = feature_dir / f"coco_{CAPTION_REVISION[:8]}_{clip_cache_tag}"
instruction_cache_path = feature_dir / f"vqav2_train_{INSTRUCTION_REVISION[:8]}_{clip_cache_tag}"
validation_cache_path = feature_dir / f"vqav2_validation_{VALIDATION_REVISION[:8]}_{clip_cache_tag}"
print("artifact directory:", artifact_dir)

## 2. Did we actually load all the data?

This is where the downloading starts: about 13.5 GB for VQAv2 on the first run. Before we spend hours of GPU time, let's make sure we got what we think we got. Every load comes with an `assert` on the row count. That's a cheap way to catch a truncated download or the wrong split before it costs us anything.

### 2.1 Load every caption row

Notice there's no `.select(...)` and no slicing here. The `assert` turns "Stage 1 uses all 5,000 captions" from a promise into something the code actually checks.

In [ ]:
caption_source = load_dataset(  # the whole train split, no slicing
    CAPTION_DATASET_ID,
    split="train",
    revision=CAPTION_REVISION,
)
assert len(caption_source) == 5_000
print("caption rows:", len(caption_source))

5,000 caption rows, but how many actual photos? Let's count the distinct `cocoid` values (COCO's image ID). That number is the reason our cache is keyed by image, not by caption.

In [ ]:
caption_image_count = len(set(caption_source["cocoid"]))  # several captions can share one cached patch grid
print("distinct caption images:", caption_image_count)

That should say 1,000, so each photo comes with about five captions.

### 2.2 Load the grouped VQAv2 training data

We download the whole thing instead of streaming it, because we'll keep coming back to it for features and tokens. Remember, each row is one image, and its `texts` field can hold several turns.

In [ ]:
instruction_source = load_dataset(  # all VQAv2 turns, grouped by image
    INSTRUCTION_DATASET_ID,
    INSTRUCTION_CONFIG,
    split="train",
    revision=INSTRUCTION_REVISION,
)
assert len(instruction_source) == 82_772
print("grouped instruction images:", len(instruction_source))

> **Heads up:** 82,772 rows is *not* how many training examples we have. Each row can hold several turns. So we add up the lengths of all the nested `texts` lists and check for 443,757. This catches a sneaky bug: accidentally training on only the first question for each image.

In [ ]:
instruction_turn_count = sum(len(turns) for turns in instruction_source["texts"])  # every nested turn becomes its own example
assert instruction_turn_count == 443_757
print("instruction turns:", f"{instruction_turn_count:,}")

### 2.3 Load the held-out records

Validation is simpler: one question per row. It gets its own pinned revision and, later, its own cache, so the line between "training" and "evaluation" stays visible both in the code and on disk.

In [ ]:
validation_source = load_dataset(  # all 1,000 rows, all held out
    VALIDATION_DATASET_ID,
    split="validation",
    revision=VALIDATION_REVISION,
)
assert len(validation_source) == 1_000
print("validation rows:", len(validation_source))

### 2.4 What does each stage ask the model to predict?

Stage 1 always gets the same generic request (describe the image) and learns to produce a human-written caption. Stage 2 gets a real question and learns its short VQAv2 answer. Let's grab one record from each source, before any tokenizing happens.

In [ ]:
caption_preview = caption_source[0]  # one record from each source, images still attached
instruction_preview = instruction_source[0]
instruction_turn_preview = instruction_preview["texts"][0]
validation_preview = validation_source[0]

Look at the two panels side by side. What stays the same, and what changes? Both stages pair a photo with some target text. Only Stage 2 adds a specific question.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # one panel per paper stage
axes[0].imshow(caption_preview["image"].convert("RGB"))
axes[0].set_title("Stage 1\n" + fill(caption_preview["caption"].strip(), 36), fontsize=9)
axes[1].imshow(instruction_preview["images"][0].convert("RGB"))
turn_title = instruction_turn_preview["user"] + "\n" + instruction_turn_preview["assistant"]
axes[1].set_title("Stage 2\n" + fill(turn_title, 38), fontsize=9)  # wrap the text so it stays next to its image
for axis in axes:
    axis.axis("off")
fig.tight_layout()
plt.show()

And here are the raw strings. These are the exact targets the model learns to predict, token by token. Keep them in mind; they make the loss numbers later a lot easier to picture.

In [ ]:
print("caption target:", caption_preview["caption"].strip())  # real dataset text, nothing hand-written
print("VQAv2 user:   ", instruction_turn_preview["user"])
print("VQAv2 answer: ", instruction_turn_preview["assistant"])

**TODO:** One Cauldron row can hold several turns about the same photo. Change `my_row` to peek at other images. How many questions does each photo get?

In [ ]:
# TODO: try other grouped rows (0 to 82,771)
my_row = 0
my_turns = instruction_source[my_row]["texts"]
print("turns for this image:", len(my_turns))
for my_turn in my_turns:
    print(" ", my_turn["user"], "->", my_turn["assistant"])

Every one of those turns will become its own training example, and all of them will point back at the same cached image.

## 3. How does an image get *into* a language model?

*LLaVA paper, Section 4.1, Architecture.*

Here's something that trips people up: we're **not** using CLIP's text embedding space at all. The destination is Qwen's **word-embedding space**, the table of vectors Qwen looks up for each token before its transformer runs. A causal language model can only read vectors of its own embedding width, so that's where the image has to land.

Here's the interface we're building:

`CLIP patch [768] → linear projection → one Qwen input position [896]`

The paper writes this as $\mathbf{H}_v = \mathbf{W} \cdot \mathbf{Z}_v$: the CLIP features $\mathbf{Z}_v$ times a learned matrix $\mathbf{W}$ give "visual tokens" $\mathbf{H}_v$ that are as wide as Qwen's word embeddings. Our `nn.Linear` also adds a bias, but the idea is the same.

Here's the paper's Figure 1, which shows the whole model on one page:

<img src="../assets/diagrams/notebook9/llava_fig1.png" alt="LLaVA network architecture" width="700">

*Figure 1 from Liu et al. (2023).*

Reading it from the bottom left:

- $\mathbf{X}_v$ is the image. It goes through the **Vision Encoder** (frozen CLIP, Section 3.1) and comes out as patch features $\mathbf{Z}_v$, which we cache in Section 4.
- **Projection W** is our `nn.Linear` connector from Section 3.3. It turns $\mathbf{Z}_v$ into visual tokens $\mathbf{H}_v$.
- $\mathbf{X}_q$ is the question. It becomes ordinary word embeddings $\mathbf{H}_q$.
- Both sets of tokens go into the **Language Model** $f_\phi$. The paper uses Vicuna; we use the much smaller Qwen2.5-0.5B.
- $\mathbf{X}_a$ is the answer the model writes. It's also the only part of the sequence that gets a loss, which we set up in Section 5.

### 3.1 Load the CLIP vision tower

We load the whole published CLIP checkpoint, keep only its vision transformer, and let the rest go (we don't need the text side). The matching image processor brings the resize-and-normalize recipe, just like in Notebook 01.

In [ ]:
image_processor = CLIPImageProcessor.from_pretrained(CLIP_ID, revision=CLIP_REVISION)
clip_model = CLIPModel.from_pretrained(CLIP_ID, revision=CLIP_REVISION).to(device)  # loading the full checkpoint avoids partial-load warnings
vision_encoder = clip_model.vision_model
del clip_model  # we only keep the vision tower

Now we freeze CLIP, before caching anything. If its weights could change later, every feature we saved to disk would go stale. You should see a trainable count of zero.

In [ ]:
vision_encoder.eval()  # frozen for good, so its outputs can be cached once and reused
for parameter in vision_encoder.parameters():
    parameter.requires_grad_(False)
print("trainable vision parameters:", sum(p.numel() for p in vision_encoder.parameters() if p.requires_grad))

### 3.2 Load Qwen and give it an image marker

Qwen brings four things: the tokenizer, the word-embedding table, the causal transformer, and the output head that turns hidden states back into token scores. The tokenizer and the weights come from the same pinned revision.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LM_ID, revision=LM_REVISION)  # tokenizer and weights from the same revision
language_model = AutoModelForCausalLM.from_pretrained(
    LM_ID, revision=LM_REVISION, dtype=torch.float32
).to(device)

Next, a new special token: `<image>`. Think of it as a bookmark. It marks *where* in the prompt the image goes; it doesn't describe the photo at all. Right before Qwen runs, we'll pull out that one marker's embedding and put the full sequence of 49 projected patches in its place.

(Qwen's embedding table can have more rows than its tokenizer has tokens, so the `max(...)` below makes sure resizing never chops any rows off.)

In [ ]:
IMAGE_TOKEN = "<image>"  # one special token = one place to insert the image
tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
embedding_rows = max(len(tokenizer), language_model.get_input_embeddings().num_embeddings)
language_model.resize_token_embeddings(embedding_rows)  # never shrinks away Qwen's spare rows
image_token_id = tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
print("image token id:", image_token_id)

### 3.3 Build the paper's linear connector

We read both widths straight from the loaded configs instead of typing them in. Then one `nn.Linear` maps 768 → 896. The same layer gets applied to every patch position separately, so it's really one translation rule shared by all 49 patches.

In [ ]:
vision_width = vision_encoder.config.hidden_size  # both sides of the connector come from the configs
language_width = language_model.get_input_embeddings().embedding_dim
projector = nn.Linear(vision_width, language_width).to(device)
print("connector contract:", vision_width, "→", language_width)

> **Pause and predict:** The connector is a 768 → 896 linear layer with a bias. How many parameters is that? And how does it compare with Qwen, a "0.5B" model?

In [ ]:
projector_parameters = sum(parameter.numel() for parameter in projector.parameters())  # weight + bias, all of it brand new
language_parameters = sum(parameter.numel() for parameter in language_model.parameters())
print("connector / language parameters:", f"{projector_parameters:,}", f"{language_parameters:,}")

768 × 896 + 896 = **689,024** parameters for the connector, against roughly half a billion for Qwen. Tiny. But it sits in a very important spot: it's the *only* path by which CLIP features can become something Qwen can read.

## 4. Why cache the image features, and how?

CLIP stays frozen in *both* stages, so a given image always produces exactly the same patch tensor. Why recompute it for every question? We store it once, trading some disk space for a huge cut in repeated GPU work.

> **Heads up:** The VQAv2 cache is the longest prep step, so try not to interrupt it. A cache only counts as finished once its `dataset_info.json` file shows up on disk. If you stop halfway, the next run starts that encoding over.

### 4.1 Which CLIP features do we take?

*LLaVA paper, Section 4.1, plus the ablations later in the paper.*

Not CLIP's final output. The paper tried the grid features from both before and after the last transformer layer, and the one *before* worked a bit better. Their guess: the last layer leans toward global, abstract properties, while the one before it keeps more local detail. So, following LLaVA, we take the **penultimate** (second-to-last) hidden layer and drop the class token, keeping only the patches.

For ViT-B/32 that's a 7 × 7 grid: 49 patch vectors per image, each 768 numbers wide. Same grid you met in Notebook 01.

In [ ]:
def encode_images(images):
    pixels = image_processor(images=images, return_tensors="pt").pixel_values.to(device)  # CLIP's own resize + normalize recipe
    with torch.no_grad():
        output = vision_encoder(pixel_values=pixels, output_hidden_states=True)
    # penultimate layer, class token dropped: [B, 50, 768] -> [B, 49, 768]
    return output.hidden_states[-2][:, 1:].detach()

Before writing anything, we tell Arrow exactly what each row looks like: an image ID plus a `[49, 768]` array stored as **fp16** (16-bit floats, half the disk space of the usual 32-bit ones). Declaring the shape up front lets Arrow write batch after batch to disk without holding the whole cache in RAM.

In [ ]:
patches_per_side = vision_encoder.config.image_size // vision_encoder.config.patch_size  # 224 // 32 = 7 for CLIP-B/32
patch_count = patches_per_side ** 2
feature_schema = Features({"image_id": Value("int64"), "patches": Array2D((patch_count, vision_width), "float16")})
print("cached patch shape:", (patch_count, vision_width))

### 4.2 Skip repeated photos

Caption rows (and validation rows) can reuse the same image. This helper remembers the first row where each image ID shows up, in order, so every photo gets encoded only once.

In [ ]:
def first_row_per_image(dataset, id_column):
    first_rows = OrderedDict()  # first appearance decides the cache order
    for row_index, image_id in enumerate(dataset[id_column]):
        first_rows.setdefault(int(image_id), row_index)
    return list(first_rows), list(first_rows.values())

This should print 1,000, not 5,000. And don't worry, no captions get thrown away. We're only skipping redundant CLIP work.

In [ ]:
caption_image_ids, caption_image_rows = first_row_per_image(caption_source, "cocoid")  # captions stay; we just pick one row per photo to encode
caption_unique_source = caption_source.select(caption_image_rows)
print("caption images to encode:", len(caption_unique_source))

Same deal for validation. The Cauldron already groups training turns by image, so its 82,772 rows can be encoded directly, no dedup needed.

In [ ]:
validation_image_ids, validation_image_rows = first_row_per_image(validation_source, "id_image")  # repeated validation images share one feature row
validation_unique_source = validation_source.select(validation_image_rows)
print("validation images to encode:", len(validation_unique_source))

### 4.3 Turn a batch of images into cache rows

This helper converts each image to RGB, runs CLIP, casts the patches to fp16, and moves them back to the CPU. It also returns an image ID for each row. That's how a text record will find its patches later.

In [ ]:
def feature_rows(images, image_ids):
    rgb_images = [image.convert("RGB") for image in images]  # some photos are grayscale; force 3 channels
    patches = encode_images(rgb_images).cpu().numpy().astype(np.float16)
    return {"image_id": np.asarray(image_ids, dtype=np.int64), "patches": patches}

Each dataset stores its images a little differently, so each gets a tiny adapter. Captions are easy: one `image` column and one `cocoid` column.

In [ ]:
def encode_caption_batch(batch):
    return feature_rows(batch["image"], batch["cocoid"])  # each selected row is one distinct COCO image

The Cauldron wraps each image in a one-item list. And instead of a COCO ID, we use the grouped row index as the cache ID. Every question nested in that row will point back to it.

In [ ]:
def encode_instruction_batch(batch, indices):
    images = [image_group[0] for image_group in batch["images"]]  # one image per grouped row, wrapped in a list
    return feature_rows(images, indices)

Validation keeps its own `id_image` values. Training and validation lookups live in separate caches, so they can never accidentally share a row.

In [ ]:
def encode_validation_batch(batch):
    return feature_rows(batch["image"], batch["id_image"])  # held-out IDs, kept apart from training indices

### 4.4 Build each cache, or reuse it

`Dataset.map` runs our encoder over batches of images and writes the declared Arrow rows. We drop all the source columns so the JPEG bytes and leftover metadata don't get copied into the cache.

In [ ]:
def build_feature_cache(source, encoder, path, with_indices=False):
    cache = source.map(  # Arrow writes each batch to disk as it goes, not all at once in RAM
        encoder, batched=True, batch_size=CACHE_BATCH_SIZE, with_indices=with_indices,
        remove_columns=source.column_names, features=feature_schema,
        desc=f"Encoding {path.name}",
    )
    cache.save_to_disk(path)
    return cache

And here's the restart trick. If the cache folder already has Arrow's `dataset_info.json`, the cache is finished and we just load it. Otherwise we encode. One `if`, and preprocessing becomes restartable.

In [ ]:
def load_or_build_feature_cache(source, encoder, path, with_indices=False):
    if (path / "dataset_info.json").exists():  # a finished cache survives kernel restarts
        return load_from_disk(path)
    return build_feature_cache(source, encoder, path, with_indices)

Stage 1 first: one feature row per distinct COCO image. All the captions will point back to these rows.

In [ ]:
caption_cache = load_or_build_feature_cache(  # shared by all the captions of each photo
    caption_unique_source, encode_caption_batch, caption_cache_path
)
print("caption feature rows:", len(caption_cache))

Now the big one. Stage 2 encodes all 82,772 grouped images. This takes way longer than the caption cache and writes roughly 6.2 GB (82,772 × 49 × 768 numbers at 2 bytes each), plus a bit of metadata. Good time for a coffee.

In [ ]:
instruction_cache = load_or_build_feature_cache(  # row index = cache ID for all of that image's turns
    instruction_source, encode_instruction_batch, instruction_cache_path, with_indices=True
)
print("instruction feature rows:", len(instruction_cache))

The held-out cache gets built separately. Keeping it physically apart on disk mirrors the separation we want during evaluation.

In [ ]:
validation_cache = load_or_build_feature_cache(  # these images never show up in an optimizer batch
    validation_unique_source, encode_validation_batch, validation_cache_path
)
print("validation feature rows:", len(validation_cache))

## 5. How do text and image become one sequence?

*LLaVA paper, Section 4.2, the training format and loss.*

The visual features are on disk now. Next we build lightweight language records that hold just three things: a row number into the patch cache, a question, and an answer. Then we tokenize all the text once, up front.

### 5.1 Point every caption at its cached photo

A lookup table maps each original image ID to its row number in the Arrow cache. That row number is just a small integer, so it's cheap to repeat across all the captions of one photo.

In [ ]:
def cache_lookup(cache):
    return {int(image_id): index for index, image_id in enumerate(cache["image_id"])}  # image ID -> row number in the cache

caption_lookup = cache_lookup(caption_cache)
validation_lookup = cache_lookup(validation_cache)

All 5,000 caption targets make it through. We attach `patch_index` first, then drop the image bytes and the other columns we don't need. Every caption turns into a tiny conversation: the same fixed request, with the caption as the answer.

In [ ]:
DESCRIPTION_REQUEST = "Describe the image in one short sentence."  # one fixed request turns each caption into a Q&A pair
caption_indices = [caption_lookup[int(cocoid)] for cocoid in caption_source["cocoid"]]
caption_records = caption_source.add_column("patch_index", caption_indices)
caption_records = caption_records.select_columns(["patch_index", "caption"])
caption_records = caption_records.rename_column("caption", "answer")
caption_records = caption_records.add_column("question", [DESCRIPTION_REQUEST] * len(caption_records))
print("Stage 1 records:", len(caption_records))

### 5.2 Flatten every VQAv2 turn

One grouped image row can turn into several language records. This function repeats the row's patch index once per turn and splits out each user question and assistant answer.

> **Pause and predict:** How many rows will come out of this: 82,772 or 443,757?

In [ ]:
def flatten_turn_batch(batch, indices):
    patch_indices, questions, answers = [], [], []  # the output can be longer than the input batch
    for image_index, turns in zip(indices, batch["texts"]):
        patch_indices.extend([image_index] * len(turns))  # every turn reuses its image's cached grid
        questions.extend(turn["user"] for turn in turns)
        answers.extend(turn["assistant"] for turn in turns)
    return {"patch_index": patch_indices, "question": questions, "answer": answers}

Removing the grouped columns is what lets `map` change the number of rows. Then the `assert` checks that no turn got lost along the way.

In [ ]:
instruction_records = instruction_source.map(  # one row per question from here on
    flatten_turn_batch,
    batched=True,
    with_indices=True,
    remove_columns=instruction_source.column_names,
    desc="Flattening all VQAv2 turns",
)
assert len(instruction_records) == instruction_turn_count
print("Stage 2 records:", f"{len(instruction_records):,}")

443,757: one row per *turn*, not per image. From here on, every question is its own training example, shuffled independently.

Validation already has one question per row. It just needs patch indices from *its own* lookup, plus the fields we'll use for evaluation.

In [ ]:
validation_indices = [validation_lookup[int(image_id)] for image_id in validation_source["id_image"]]  # validation lookup only, never the training cache
validation_records = validation_source.add_column("patch_index", validation_indices)
validation_records = validation_records.select_columns(["patch_index", "question", "multiple_choice_answer"])
validation_records = validation_records.rename_column("multiple_choice_answer", "answer")
print("validation records:", len(validation_records))

### 5.3 Format the prompts and tokenize them once

Qwen is an instruct model, so it expects its own **chat template**: special tokens that mark where the system message, the user's turn, and the assistant's reply start and end. We use the template that ships with Qwen and put `<image>` at the start of the user turn, right before the question. The answer stays separate, so later we can hide every non-answer position from the loss.

The paper formats its training examples the same way. Here's its Table 2:

<img src="../assets/diagrams/notebook9/llava_table2.png" alt="LLaVA Table 2: the input sequence used to train the model" width="700">

*Table 2 from Liu et al. (2023).*

Only the green parts get a loss: the assistant's answers $\mathbf{X}_a$ and the `<STOP>` token that ends each one. The system message and the human's turn are there for context, but the model is never graded on them. We follow the same rule with Qwen's chat template: a little further down, every position that isn't part of the answer gets the label `-100`, which tells the loss to skip it.

In [ ]:
SYSTEM_INSTRUCTION = "Answer using the image."  # captions and VQA share one conversation frame

def format_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": f"{IMAGE_TOKEN}\n{question}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

We tokenize in batches during preprocessing. That way the training loop never touches a string; across all 27,735 Stage 2 steps it just grabs ready-made token IDs. Each answer also gets Qwen's end-of-turn token tacked on, which is how the model learns when to stop talking.

In [ ]:
def tokenize_record_batch(batch):
    prompts = [format_prompt(question) for question in batch["question"]]  # Qwen's chat template around every request
    targets = [answer.strip() + tokenizer.eos_token for answer in batch["answer"]]
    prompt_ids = tokenizer(prompts, add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(targets, add_special_tokens=False)["input_ids"]  # the end-of-turn token teaches it to stop
    return {"prompt_ids": prompt_ids, "target_ids": target_ids}

After mapping, each record only needs three fields: a patch row index, the prompt IDs, and the target IDs.

In [ ]:
def tokenize_records(records, description):
    tokenized = records.map(  # text work happens once per record, not once per training step
        tokenize_record_batch, batched=True, batch_size=1024, desc=description
    )
    return tokenized.select_columns(["patch_index", "prompt_ids", "target_ids"])

Three different sources, one shared compact format. Check the counts one more time before the expensive stuff starts.

In [ ]:
caption_tokens = tokenize_records(caption_records, "Tokenizing all captions")  # every row kept
instruction_tokens = tokenize_records(instruction_records, "Tokenizing all VQAv2 turns")
validation_tokens = tokenize_records(validation_records, "Tokenizing validation records")
print("tokenized rows:", len(caption_tokens), len(instruction_tokens), len(validation_tokens))

### 5.4 Swap one marker for 49 projected patches

This is *the* Mini-LLaVA move, the one everything else exists to support. The function:

1. finds `<image>` in the prompt,
2. embeds the text before it with Qwen's embedding table,
3. projects every CLIP patch from 768 to 896,
4. embeds the text after it,
5. glues the three pieces together.

The sequence grows by 48 positions: one marker out, 49 visual positions in.

In [ ]:
def splice_visual_embeddings(patches, prompt_ids):
    prompt = torch.as_tensor(prompt_ids, dtype=torch.long, device=device)  # token IDs only move to the GPU with their mini-batch
    marker = torch.where(prompt == image_token_id)[0].item()
    word_embeddings = language_model.get_input_embeddings()
    before_image = word_embeddings(prompt[:marker])
    visual_embeddings = projector(patches)  # [49, 768] -> [49, 896]
    after_image = word_embeddings(prompt[marker + 1:])
    return torch.cat((before_image, visual_embeddings, after_image))

Now the labels. We tack the answer's embeddings onto the end of that multimodal context. Then the labels get `-100` for every context position (system text, user text, the 49 image positions, even the assistant header) and the real token IDs for the answer. `-100` is PyTorch's "ignore this position" value for cross-entropy, so the loss only cares about the answer. The paper does the same thing: only the assistant's answer tokens count toward the loss.

In [ ]:
def make_example(patches, prompt_ids, target_ids):
    target = torch.as_tensor(target_ids, dtype=torch.long, device=device)
    context = splice_visual_embeddings(patches, prompt_ids)  # image + user text: everything the answer gets to see
    answer_embeddings = language_model.get_input_embeddings()(target)
    inputs = torch.cat((context, answer_embeddings))
    ignored = torch.full((context.shape[0],), -100, device=device)  # -100: no loss on any context position
    labels = torch.cat((ignored, target))
    return inputs, labels

> **Heads up:** Wait, the answer is *in* the input? Isn't that cheating? Nope. Qwen shifts the labels internally, so each position is trained to predict the token one step to its right. And causal attention means a position can only look left, never right. So no position ever gets to peek at the token it's supposed to predict.

`make_batch` builds a whole mini-batch. Sequences have different lengths, so it pads them on the right: zero vectors for the inputs, `-100` for the labels, and zeros in the attention mask so Qwen ignores the padding.

In [ ]:
def make_batch(patches, prompts, targets):
    examples = [make_example(*items) for items in zip(patches, prompts, targets)]  # zip keeps each image paired with its own text
    inputs = pad_sequence([item[0] for item in examples], batch_first=True)
    labels = pad_sequence([item[1] for item in examples], batch_first=True, padding_value=-100)
    masks = [torch.ones(item[0].shape[0], dtype=torch.long, device=device) for item in examples]
    attention_mask = pad_sequence(masks, batch_first=True)  # zeros only show up after a sequence ends
    return {"inputs_embeds": inputs, "labels": labels, "attention_mask": attention_mask}

### 5.5 Shuffle into mini-batches

Token sequences have different lengths, so the collator just keeps them as plain lists. Padding only happens later, in `make_batch`, *after* the patches are spliced in, because that's when we finally know each sequence's real length.

In [ ]:
def collate_tokenized(records):
    return {  # only small indices and token lists go through the loader
        "patch_index": torch.tensor([row["patch_index"] for row in records]),
        "prompt_ids": [row["prompt_ids"] for row in records],
        "target_ids": [row["target_ids"] for row in records],
    }

Stage 1 and Stage 2 each get their own seeded random generator, so each shuffle is repeatable on its own. Validation stays in its original order and keeps its last, smaller batch.

In [ ]:
caption_generator = torch.Generator().manual_seed(SEED)  # each stage gets its own repeatable shuffle
instruction_generator = torch.Generator().manual_seed(SEED + 1)
caption_loader = DataLoader(caption_tokens, batch_size=ALIGNMENT_BATCH_SIZE, shuffle=True, collate_fn=collate_tokenized, generator=caption_generator)
instruction_loader = DataLoader(instruction_tokens, batch_size=INSTRUCTION_BATCH_SIZE, shuffle=True, collate_fn=collate_tokenized, generator=instruction_generator)
validation_loader = DataLoader(validation_tokens, batch_size=INSTRUCTION_BATCH_SIZE, shuffle=False, collate_fn=collate_tokenized)
print("steps per epoch:", len(caption_loader), len(instruction_loader))

That's 157 steps for Stage 1 (5,000 ÷ 32, rounded up) and 27,735 for Stage 2 (443,757 ÷ 16, rounded up). Yep, Stage 2 is where the hours go.

One more helper. Each mini-batch reads only the feature rows it asks for from Arrow, not the whole column. They're stored as fp16 on disk and get converted to the projector's dtype (fp32 here) on their way to the GPU.

In [ ]:
def fetch_cached_patches(cache, patch_indices):
    rows = cache[patch_indices.tolist()]["patches"]  # read just these rows, not the whole column
    return torch.as_tensor(np.asarray(rows), dtype=projector.weight.dtype, device=device)

### 5.6 Check one sequence before training

This preflight check is worth a careful read. If something's off here, a multi-hour run would only make the mistake expensive. In the output, confirm three things:

1. one prompt marker turns into 49 visual positions,
2. the combined width is Qwen's embedding width (896),
3. the number of supervised positions equals the number of answer tokens.

> **Pause and predict:** Say the prompt has *P* token IDs (including the one `<image>`) and the answer has *T*. How many positions will the combined sequence have?

In [ ]:
preflight_record = caption_tokens[0]  # one real record to check the layout
preflight_patches = fetch_cached_patches(caption_cache, torch.tensor([preflight_record["patch_index"]]))[0]
with torch.no_grad():  # just checking shapes, no training graph needed
    preflight_inputs, preflight_labels = make_example(preflight_patches, preflight_record["prompt_ids"], preflight_record["target_ids"])
print("prompt / visual positions:", len(preflight_record["prompt_ids"]), preflight_patches.shape[0])
print("combined positions × width:", tuple(preflight_inputs.shape))
print("supervised answer positions:", preflight_labels.ne(-100).sum().item())
del preflight_patches, preflight_inputs, preflight_labels

It's *P* − 1 + 49 + *T*: the marker leaves, 49 patches arrive, and the answer goes on the end. Only those *T* answer positions carry real labels; everything else is `-100`.

**TODO:** Shapes are nice, but what does the sequence actually *say*? The cell below decodes the pieces around the image slot, so you can see exactly where the patches land. Change `my_row`, or point `my_tokens` at `instruction_tokens` or `validation_tokens`.

In [ ]:
# TODO: try instruction_tokens or validation_tokens, and other rows
my_tokens = caption_tokens
my_row = 0

my_record = my_tokens[my_row]
my_prompt = my_record["prompt_ids"]
my_marker = my_prompt.index(image_token_id)  # where <image> sits in the prompt
print("text before the image:", repr(tokenizer.decode(my_prompt[:my_marker])))
print(f"image slot:            {patch_count} projected patches replace token id {image_token_id}")
print("text after the image: ", repr(tokenizer.decode(my_prompt[my_marker + 1:])))
print("answer (supervised):  ", repr(tokenizer.decode(my_record["target_ids"])))
print("total positions:", len(my_prompt) - 1 + patch_count + len(my_record["target_ids"]))

See how the image sits inside the user turn, after the system message and the user header, right before the question? And notice the assistant header (the bit that says "your turn, model") is part of the *prompt*, not the answer. So it never gets a label; the model only gets graded on what it says after it.

## 6. Stage 1: how do we teach the connector to speak Qwen?

*LLaVA paper, Section 4.2, "Stage 1: Pre-training for Feature Alignment."*

In Stage 1, both pretrained models stay frozen. All 5,000 captions provide answer-token loss, but the linear connector is the only thing that can respond to it. The paper describes this stage as training a compatible visual tokenizer for the frozen language model. In plain words: the projector learns to put image patches somewhere in Qwen's embedding space where Qwen can make sense of them.

### 6.1 Freeze Qwen

`eval()` switches off training-only behavior like dropout. And here's a subtle but important point: even with Qwen frozen, gradients still flow *through* Qwen's operations, back to the projected image inputs. They just don't get stored in any of Qwen's weights. That's how the connector, which sits *before* Qwen, can learn from a loss computed *after* it.

In [ ]:
for parameter in language_model.parameters():  # Stage 1: no Qwen weight changes
    parameter.requires_grad_(False)
language_model.eval()
projector.train()
print("trainable connector parameters:", sum(p.numel() for p in projector.parameters() if p.requires_grad))

### 6.2 One epoch, nothing hidden

This function does six things, all in plain sight: fetch the patches, build the sequences, compute the next-token loss, clear old gradients, backpropagate, and step the optimizer. Notice it has no idea which stage it's in. That's decided entirely by which parameters can take gradients and which ones the optimizer holds.

In [ ]:
def train_one_epoch(loader, feature_cache, optimizer, description):
    losses = []  # raw per-batch losses; we smooth them later, only for plotting
    for records in tqdm(loader, desc=description):
        patches = fetch_cached_patches(feature_cache, records["patch_index"])
        model_batch = make_batch(patches, records["prompt_ids"], records["target_ids"])
        loss = language_model(**model_batch, use_cache=False).loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()  # requires_grad + the optimizer decide who actually learns from this
        optimizer.step()
        losses.append(loss.item())
    return losses

The Stage 1 optimizer only holds the projector's parameters. One epoch visits every caption once, in shuffled mini-batches, so the update count at the end should match the 157 steps from earlier. It's a short wait compared with Stage 2.

In [ ]:
alignment_optimizer = torch.optim.AdamW(projector.parameters(), lr=ALIGNMENT_LR)  # only the projector: no CLIP, no Qwen
alignment_losses = []
for epoch in range(ALIGNMENT_EPOCHS):
    epoch_losses = train_one_epoch(caption_loader, caption_cache, alignment_optimizer, f"alignment epoch {epoch + 1}")
    alignment_losses.extend(epoch_losses)
print("alignment updates:", len(alignment_losses))

Let's save the aligned connector before we change who's allowed to learn. This file is the paper's Stage 1 result, and it's a clean fallback if Stage 2 gets interrupted.

In [ ]:
stage1_path = checkpoint_dir / "stage1_projector.pt"  # just the new bridge's weights
torch.save(projector.state_dict(), stage1_path)
print("saved:", stage1_path)

### 6.3 Is alignment working?

Raw mini-batch loss jumps around a lot, because captions differ in content and length. Averaging over blocks of steps makes the overall trend readable. Smoothing only changes the picture, not the training.

In [ ]:
def block_means(values, width):
    return [  # each step lands in exactly one block
        sum(values[start:start + width]) / len(values[start:start + width])
        for start in range(0, len(values), width)
    ]

If the curve heads down, that supports one modest conclusion: the connector is getting more useful for predicting these captions. It doesn't show question answering yet, and it doesn't show generalization.

In [ ]:
alignment_curve = block_means(alignment_losses, ALIGNMENT_SMOOTHING)  # each point averages nearby steps
fig, axis = plt.subplots(figsize=(6.5, 3.5))
axis.plot(np.arange(len(alignment_curve)) * ALIGNMENT_SMOOTHING + 1, alignment_curve)
axis.set(xlabel="optimizer step", ylabel="causal-language loss", title="Stage 1: all caption pairs")
axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 7. Stage 2: how do we turn it into an assistant?

*LLaVA paper, Section 4.2, "Stage 2: Fine-tuning End-to-End."*

CLIP stays frozen, which keeps our feature cache valid. But now the connector *and* Qwen learn together, across all 443,757 question–answer records.

At batch size 16 that's 27,735 optimizer steps. This is where the real compute goes, and it can take several hours on a single GPU. Totally normal to let it run while you go do something else.

### 7.1 Change who gets to learn

Unfreezing Qwen is what makes this the paper's second stage. The printed trainable count shows the jump. And CLIP? Still not on the list.

In [ ]:
for parameter in language_model.parameters():  # Stage 2: Qwen's weights can now adapt to visual inputs
    parameter.requires_grad_(True)
language_model.train()
projector.train()
instruction_parameters = list(projector.parameters()) + list(language_model.parameters())
print("Stage 2 trainable parameters:", f"{sum(p.numel() for p in instruction_parameters):,}")

### 7.2 Get a held-out baseline first

Before any VQAv2 training, we measure validation loss: after caption alignment, but before instruction tuning. That gives us a "before" number to compare against.

One detail: each batch is weighted by how many answer tokens it has. Otherwise the small last batch would count as much as a full one.

In [ ]:
def validation_step(records, feature_cache):
    patches = fetch_cached_patches(feature_cache, records["patch_index"])
    model_batch = make_batch(patches, records["prompt_ids"], records["target_ids"])
    output = language_model(**model_batch, use_cache=False)  # no optimizer anywhere in here
    supervised = model_batch["labels"].ne(-100).sum().item()
    return output.loss.item() * supervised, supervised

The evaluator walks through all 1,000 held-out rows with gradients off. It adds up the total **negative log-likelihood** (the per-token loss, summed instead of averaged) and the token count, then divides once at the very end.

In [ ]:
def evaluate_loss(loader, feature_cache, description):
    language_model.eval()
    projector.eval()
    total_nll, token_count = 0.0, 0  # token totals keep uneven batches comparable
    with torch.no_grad():
        for records in tqdm(loader, desc=description):
            batch_nll, batch_tokens = validation_step(records, feature_cache)
            total_nll += batch_nll
            token_count += batch_tokens
    return total_nll / token_count

Record the baseline now, before a single instruction update. We'll reuse the same loader and the same calculation afterward, so the two numbers are directly comparable.

In [ ]:
validation_before = evaluate_loss(validation_loader, validation_cache, "validation before Stage 2")  # held-out rows, untouched by training
print("validation loss before:", round(validation_before, 4))

### 7.3 Run the full instruction epoch

The optimizer now holds both Qwen and the connector, at Stage 2's much smaller learning rate. The progress bar should cover every shuffled VQAv2 turn exactly once.

> **Heads up:** This is the cell you leave running. Nothing from Stage 2 gets saved to disk until the save cell in Section 7.5. If you stop partway, the only checkpoint on disk is the Stage 1 projector from Section 6.

In [ ]:
instruction_optimizer = torch.optim.AdamW(instruction_parameters, lr=INSTRUCTION_LR)  # projector + Qwen; CLIP still left out
instruction_losses = []
language_model.train()
projector.train()
for epoch in range(INSTRUCTION_EPOCHS):
    epoch_losses = train_one_epoch(instruction_loader, instruction_cache, instruction_optimizer, f"instruction epoch {epoch + 1}")
    instruction_losses.extend(epoch_losses)
print("instruction updates:", len(instruction_losses))

### 7.4 Did held-out loss go down?

If validation loss is lower after training, the model's answer-token predictions improved on this sample. If it went *up*, that tells you something too: maybe overfitting, a learning rate that's too aggressive, or a mismatch between the training and validation data.

In [ ]:
validation_after = evaluate_loss(validation_loader, validation_cache, "validation after Stage 2")  # no training data in this measurement
print("validation loss before → after:", round(validation_before, 4), "→", round(validation_after, 4))

The Stage 2 curve averages every 200 mini-batches for display. Look at its overall direction and how bumpy it is, but don't read general VQA quality off a training loss alone.

In [ ]:
instruction_curve = block_means(instruction_losses, INSTRUCTION_SMOOTHING)  # smoothing is for display only
fig, axis = plt.subplots(figsize=(6.5, 3.5))
axis.plot(np.arange(len(instruction_curve)) * INSTRUCTION_SMOOTHING + 1, instruction_curve)
axis.set(xlabel="optimizer step", ylabel="causal-language loss", title="Stage 2: all VQAv2 turns")
axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()

### 7.5 Save everything we need to rebuild the model

Qwen and its tokenizer get saved in the standard Transformers folder format. The connector gets its own file. CLIP doesn't need saving at all: it never changed, so the pinned checkpoint at the top of the notebook is all you need.

In [ ]:
final_model_dir = checkpoint_dir / "final_language_model"  # these files + the pinned CLIP = the whole trained system
language_model.save_pretrained(final_model_dir, safe_serialization=True)
tokenizer.save_pretrained(final_model_dir)
torch.save(projector.state_dict(), checkpoint_dir / "final_projector.pt")
print("saved final model:", final_model_dir)

We also save a training summary, with the revision strings and the full loss histories, right next to the weights. If two runs ever disagree, this tells you whether the data, the starting point, or the optimization changed.

In [ ]:
training_summary = {  # revision strings make later comparisons easy to audit
    "caption_revision": CAPTION_REVISION,
    "instruction_revision": INSTRUCTION_REVISION,
    "validation_revision": VALIDATION_REVISION,
    "alignment_losses": alignment_losses,
    "instruction_losses": instruction_losses,
    "validation_before": validation_before,
    "validation_after": validation_after,
}
torch.save(training_summary, checkpoint_dir / "training_summary.pt")

## 8. Can it answer a question it's never seen?

Time for the fun part. For generation, the model gets the prompt and the image patches, but no reference answer. We'll decode one answer deterministically to check the whole trained path end to end. Then we'll swap *only* the image, as a basic test of whether the model is actually looking.

These checks are our own, by the way. They're not from the paper.

The image swap is a homemade version of a bigger idea: checking whether a model actually looks before it answers. One well-known benchmark, POPE, asks yes/no questions about objects that are and aren't in the picture:

<img src="../assets/diagrams/notebook9/book_fig5_19.png" alt="The POPE pipeline for measuring object hallucination" width="750">

*Figure 5.19 from [the book](https://arxiv.org/abs/2605.07544): the POPE pipeline, from Li et al. (2023).*

A model that answers "yes" to "Is there a dog in the image?" when there's no dog is **hallucinating**: it's answering from habit instead of from the picture. Keep that in mind for the TODOs at the end.

### 8.1 Build prompt-only inputs

Unlike training, there are no answer embeddings and no labels here. Just the spliced prompt, plus an attention mask that covers all of it.

In [ ]:
def generation_inputs(patches, prompt_ids):
    context = splice_visual_embeddings(patches, prompt_ids)  # prompt + patches only: no answer, no labels
    attention_mask = torch.ones((1, context.shape[0]), dtype=torch.long, device=device)
    return context.unsqueeze(0), attention_mask

We'll use the first validation record. It never went into either optimizer. Its patches come from the separate validation cache, and its prompt IDs from validation tokenization.

**TODO:** Once you've seen this one, change `probe_index` (0 to 999) and re-run the cells down to the end of Section 8. Generation only takes a moment, so try as many as you like.

In [ ]:
probe_index = 0  # TODO: try other held-out rows (0 to 999)
probe_record = validation_records[probe_index]
probe_tokens = validation_tokens[probe_index]
probe_patches = fetch_cached_patches(validation_cache, torch.tensor([probe_record["patch_index"]]))[0]
probe_inputs, probe_mask = generation_inputs(probe_patches, probe_tokens["prompt_ids"])

### 8.2 Decode one answer, no randomness

**Greedy decoding** just picks the highest-scoring token at every step. No sampling means no randomness, so when we swap the image later, the image is the *only* thing that changes.

In [ ]:
language_model.eval()  # done training: back to eval mode
with torch.no_grad():
    generated_ids = language_model.generate(
        inputs_embeds=probe_inputs, attention_mask=probe_mask,
        max_new_tokens=24, do_sample=False, repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
    )
generated_answer = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

Look at the photo before you read the model's answer, and come up with your own short answer first. It makes it way easier to judge both the reference and the model.

In [ ]:
fig, axis = plt.subplots(figsize=(6, 4))  # the held-out photo, with its question as the title
axis.imshow(validation_source[probe_index]["image"].convert("RGB"))
axis.axis("off")
axis.set_title(fill(probe_record["question"], 52))
fig.tight_layout()
plt.show()

Now compare three strings: the question, the dataset's reference answer, and what our model generated. An exact match is nice, but it's not the only good outcome. Short VQA answers can be worded differently and still be right ("2" vs. "two"). And remember, this is one record. It's a sanity check, not a benchmark.

In [ ]:
print("question: ", probe_record["question"])  # the reference comes straight from the untouched validation set
print("reference:", probe_record["answer"])
print("generated:", generated_answer)

### 8.3 Same question, different photo

Here's a simple but telling test. We keep the exact same question tokens but feed in a genuinely different validation photo. If the answer *never* changes when the image does, the model might be leaning on language habits (whatever answer usually follows a question like that) instead of the picture.

> **Pause and predict:** Will the answer change? Think about what the question asks, and whether it even makes sense for some random other photo.

In [ ]:
wrong_index = next(index for index, row in enumerate(validation_records) if row["patch_index"] != probe_record["patch_index"])  # first row with a genuinely different photo
wrong_record = validation_records[wrong_index]
wrong_patches = fetch_cached_patches(validation_cache, torch.tensor([wrong_record["patch_index"]]))[0]
wrong_inputs, wrong_mask = generation_inputs(wrong_patches, probe_tokens["prompt_ids"])
with torch.no_grad():
    wrong_ids = language_model.generate(
        inputs_embeds=wrong_inputs, attention_mask=wrong_mask,
        max_new_tokens=24, do_sample=False, repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
    )

Read the two answers side by side.

In [ ]:
wrong_image_answer = tokenizer.decode(wrong_ids[0], skip_special_tokens=True).strip()  # same question, different image
print("correct-image answer:", generated_answer)
print("wrong-image answer:  ", wrong_image_answer)

If they're different, that's evidence the image *can* steer generation. It doesn't prove the new answer is right, and it doesn't prove the model is reliably grounded across the dataset. If they're the same, that's not automatically a failure either: some questions have the same likely answer for lots of photos (plenty of yes/no questions, say). One swap is a hint, not a verdict. TODO 2 below lets you run a few more.

## Recap

That's the whole trip, and the end of the course. Here's what one held-out question goes through:

```text
photo
   ↓ CLIP ViT-B/32 (frozen), penultimate layer, class token dropped
[49, 768] patch grid                       cached once per image, fp16 on disk
   ↓ linear projector 768 -> 896           the only brand-new weights
[49, 896] projected patches
   ↓ spliced in where <image> was
[system + user header | 49 patches | question + assistant header | answer]
   ↓ Qwen2.5-0.5B-Instruct, next-token loss on answer tokens only
generated answer
```

And the two stages:

| | Stage 1: feature alignment | Stage 2: instruction tuning |
|---|---|---|
| Data | all 5,000 COCO captions | all 443,757 VQAv2 turns |
| Learns | projector only | projector + Qwen |
| Frozen | CLIP and Qwen | CLIP |
| Optimizer steps | 157 | 27,735 |

**The big ideas:**

- Mini-LLaVA isn't an image model and a text model glued together at the output. It's **one causal sequence** in Qwen's embedding space: system and user tokens, projected CLIP patches, and assistant tokens.
- The connector is tiny, a single linear layer, but it's the only bridge from CLIP's space into Qwen's.
- The `-100` mask decides **where the loss is measured**. `requires_grad` and the optimizer decide **who learns from it**.
- Freezing CLIP in both stages is what makes caching its features safe.
- Our run keeps the paper's align-then-instruction-tune logic, but with smaller models and VQAv2-only instruction data, it's a narrower system. Treat the validation loss, the held-out answer, and the image swap as checks of *this* implementation, not as a claim that it reproduces LLaVA's results.

### Check your understanding

1. Why is it safe to compute the CLIP features once and reuse them in both stages?
2. In Stage 1, none of Qwen's weights are trainable. So why does `loss.backward()` still have to run through Qwen?
3. A prompt has 40 token IDs, including one `<image>`, and the answer has 3 tokens (end-of-turn token included). How many positions does the training sequence have, and how many of them have a label other than `-100`?

<details>
<summary>Show answers</summary>

1. CLIP is frozen (no gradients, eval mode) in both stages, so the same image always gives exactly the same patches. If CLIP were trained, the saved features would go stale the moment its weights changed.
2. The projector sits *before* Qwen, but the loss is computed from Qwen's output. To reach the projector, the gradient has to flow back through all of Qwen's operations. Qwen's weights just don't keep any of it.
3. 40 − 1 + 49 + 3 = **91** positions. Only the **3** answer positions have real labels; the other 88 are `-100`.

</details>

## Try it yourself

Everything here reuses the model you just trained and the cached features. No retraining, and each cell should take seconds. (Run Section 8 first, so the model is back in eval mode and `probe_*` exists.)

### TODO 1: Ask your own question

The first cell below wraps the greedy decoding from Section 8.2 into a little helper, plus one that fetches the cached patches for any held-out row. Then it asks *your* question about a held-out photo. Look at the photo, change `my_index` and `my_question`, and re-run.

In [ ]:
def my_generate(patches, prompt_ids, max_new_tokens=24):
    # same greedy decoding as Section 8.2, wrapped up so we can reuse it
    my_inputs, my_mask = generation_inputs(patches, prompt_ids)
    with torch.no_grad():
        my_ids = language_model.generate(
            inputs_embeds=my_inputs, attention_mask=my_mask,
            max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.0,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(my_ids[0], skip_special_tokens=True).strip()


def my_patches_for(index):
    # cached CLIP patches for one held-out row: [49, 768]
    return fetch_cached_patches(validation_cache, torch.tensor([validation_records[index]["patch_index"]]))[0]


# TODO: pick a held-out photo (0 to 999) and write your own question
my_index = 0
my_question = "What is the main thing in this picture?"

language_model.eval()
my_prompt_ids = tokenizer(format_prompt(my_question), add_special_tokens=False)["input_ids"]
my_answer = my_generate(my_patches_for(my_index), my_prompt_ids)

fig, axis = plt.subplots(figsize=(5, 4))
axis.imshow(validation_source[my_index]["image"].convert("RGB"))
axis.axis("off")
axis.set_title(fill("Q: " + my_question, 52) + "\n" + fill("A: " + my_answer, 52), fontsize=10)
fig.tight_layout()
plt.show()

<details>
<summary>Hint</summary>

Stage 2 trained on VQAv2, so the model's comfort zone is short questions with short answers: yes/no ("Is there a dog?"), counting ("How many people are there?"), and colors ("What color is the bus?"). Try one of each.

Then try `my_question = DESCRIPTION_REQUEST`, the exact request from Stage 1. Does the model still write a caption after all that VQAv2 training, or does it answer in one or two words now?

</details>

> **Something to chew on:** Ask a question about something that clearly *isn't* in the photo ("What color is the cat?" on a photo with no cat). What does the model do? VQAv2 questions were written about what's actually in each photo, so the model has rarely been asked about something that isn't there.

### TODO 2: Swap images, properly

Section 8.3 swapped one image. One swap is a weak test, so let's do a few. This cell keeps one question fixed and asks it about several photos.

In [ ]:
# TODO: change the question, or which photos to try (each 0 to 999)
my_swap_question = probe_record["question"]
my_swap_indices = [0, 250, 500, 750]

my_swap_prompt = tokenizer(format_prompt(my_swap_question), add_special_tokens=False)["input_ids"]
fig, axes = plt.subplots(1, len(my_swap_indices), figsize=(4 * len(my_swap_indices), 4))
for my_axis, my_i in zip(np.atleast_1d(axes), my_swap_indices):
    my_swap_answer = my_generate(my_patches_for(my_i), my_swap_prompt)
    my_axis.imshow(validation_source[my_i]["image"].convert("RGB"))
    my_axis.set_title(fill(f"row {my_i}: {my_swap_answer}", 30), fontsize=10)
    my_axis.axis("off")
fig.suptitle(fill("Q: " + my_swap_question, 80))
fig.tight_layout()
plt.show()

Do the answers follow the photos? Try a question that has an obvious, image-dependent answer, like "How many people are in the picture?" or "What color is the car?", and compare it with a yes/no question.

<details>
<summary>Hint</summary>

Some validation rows share the same photo (that's why we deduplicated them in Section 4.2). If two rows show the same image, they'll get the same answer, of course. Spread your indices out to avoid that.

</details>

### TODO 3: Blindfold the model

What if there's no real image at all? Here we feed all-zero "patches" instead of the real CLIP features. A linear layer applied to zeros just outputs its bias, so all 49 image positions become the same vector. That's a crude control (zeros look nothing like real CLIP features), but it's a quick way to see how much the answer depends on the picture.

In [ ]:
# TODO: try other held-out rows, or swap zeros for noise: torch.randn_like(probe_patches)
my_blind_index = probe_index
my_blank_patches = torch.zeros_like(probe_patches)  # [49, 768] of zeros: the projector outputs only its bias

my_blind_prompt = validation_tokens[my_blind_index]["prompt_ids"]
print("question:      ", validation_records[my_blind_index]["question"])
print("reference:     ", validation_records[my_blind_index]["answer"])
print("with the image:", my_generate(my_patches_for(my_blind_index), my_blind_prompt))
print("blindfolded:   ", my_generate(my_blank_patches, my_blind_prompt))

If the blindfolded answer is often the same as the real one, the model is answering from the question alone for those cases. Try a handful of rows. Which kinds of questions survive the blindfold?

### TODO 4: What do projected patches look like to Qwen?

After the projector, each patch is an 896-number vector living in the same space as Qwen's word embeddings. So which actual *words* is it closest to? This cell compares one projected patch with every token embedding in Qwen's vocabulary (cosine similarity) and prints the top 10.

In [ ]:
# TODO: pick a patch (0 to 48, read left to right, top to bottom on the 7 x 7 grid)
my_patch_number = 24  # 24 is the center patch

with torch.no_grad():
    my_projected = torch.nn.functional.normalize(projector(probe_patches), dim=-1)  # [49, 896]
    my_vocab = torch.nn.functional.normalize(language_model.get_input_embeddings().weight[:len(tokenizer)], dim=-1)  # [vocab, 896]
    my_scores = my_projected[my_patch_number] @ my_vocab.T  # cosine similarity with every token
    my_top = my_scores.topk(10)
for my_score, my_token_id in zip(my_top.values.tolist(), my_top.indices.tolist()):
    print(f"{my_score:.3f}  {tokenizer.decode([my_token_id])!r}")

Look at both the words *and* the scores. If the scores are small and the words look random, that's a real finding, not a bug: projected patches don't have to sit next to any actual word. They only have to be *useful* to Qwen. Try a few patches. Do neighboring patches get similar lists?

---

## Where to go from here

You made it to the end of the course. You started with a 32 × 32 airplane turning into a 7 × 7 grid of patch tokens in Notebook 01, and you just spliced that same kind of grid into a real language model and taught it to answer questions. Not bad.

If you want to keep going:

- **The official [LLaVA repo](https://github.com/haotian-liu/LLaVA)** has the full-size training code, data, and checkpoints. Now that you've built the core by hand, its model code should look pretty familiar.
- **The LLaVA-1.5 paper**, [*Improved Baselines with Visual Instruction Tuning*](https://arxiv.org/abs/2310.03744), keeps the same recipe but makes a few simple changes, like swapping the linear layer for a small two-layer MLP and using a higher-resolution CLIP.
- **Run a full-size LLaVA in a few lines.** Hugging Face `transformers` has `LlavaForConditionalGeneration`, and the `llava-hf` organization on the Hub hosts converted checkpoints. Ask it the same held-out questions and compare.
- **Bigger projects** (these mean retraining, so budget the time): swap `nn.Linear` for a two-layer MLP; try CLIP ViT-B/16, which gives 14 × 14 = 196 patches per image instead of 49; or train Stage 2 on richer instruction data like LLaVA-Instruct-158K and see whether the answers get longer and more conversational.

Thanks for sticking with it all the way through.